# jia2024

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/jia2024.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们读入贾、王、司（2024）使用的微观调查数据，并对政治经济学暴露变量 PEE 取对数，以缓解右偏并便于用弹性解释。

In [ ]:
%%stata
clear
use ../datasets/Jia_Wang_Si2024.dta
// 取对数
gen lnPEE=log(PEE)

我们对 lnPEE 做去平均处理，生成离差 dm_lnPEE。

In [ ]:
%%stata
// 去平均
egen m_lnPEE=mean(lnPEE)
gen dm_lnPEE=lnPEE-m_lnPEE

我们生成户口虚拟变量与去平均后 lnPEE 的交乘项。使用离差构造交乘项后，主效应 hukou 的系数解释为在 lnPEE 处于样本均值时户口对收入的影响，而不是在 lnPEE=0（通常不在样本支撑范围内）时的影响，解释更有意义。

In [ ]:
%%stata
// 交乘项
gen hukou_dm_lnPEE=hukou*dm_lnPEE

我们把人口学控制变量存入局部宏，并估计不含任何固定效应的基准 OLS；随后用 outreg2 把结果导出，addt 选项在表格下方标注各组固定效应是否控制。

In [ ]:
%%stata
// 回归
local controls "gender nation married parents_edu"
reg lnIncome hukou_dm_lnPEE hukou lnPEE `controls'
outreg2 using jia2024.tex, replace addt(出生年份固定效应, 否, 城市固定效应, 否, 调查年份固定效应, 否, 城市×出生年份固定效应, 否)

我们逐步加入固定效应以控制遗漏变量：先控制出生年份固定效应，吸收同代人共同经历的宏观冲击。

In [ ]:
%%stata
reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(birthyear)
outreg2 using jia2024.tex, append addt(出生年份固定效应, 是, 城市固定效应, 否, 调查年份固定效应, 否, 城市×出生年份固定效应, 否)

再加入城市和调查年份固定效应，控制城市层面不随时间变化的因素以及所有城市共同的年度冲击。

In [ ]:
%%stata
reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(birthyear citycode year)
outreg2 using jia2024.tex, append addt(出生年份固定效应, 是, 城市固定效应, 是, 调查年份固定效应, 是, 城市×出生年份固定效应, 否)

最后控制城市×出生年份的交互固定效应（用 i.birthyear#i.citycode 展开为哑变量组），它吸收每个城市每一代人特有的冲击，是最严格的设定；此时单独的出生年份和城市效应被其吸收，故表中标注为“--”。比较各列中交乘项系数的变化，可以判断估计对控制变量选择的稳健性。

In [ ]:
%%stata
reghdfe lnIncome hukou_dm_lnPEE hukou lnPEE `controls', absorb(i.birthyear#i.citycode year)
outreg2 using jia2024.tex, append addt(出生年份固定效应, --, 城市固定效应, --, 调查年份固定效应, 是, 城市×出生年份固定效应, 是)